# QC summary -- all phantom orthos

Consolidated view of QC plots across all datasets and model variants.
Plots are referenced in-place (no duplication). Worst-case by_cre plots omitted.

**Canonical models** (used for simulations / downstream analysis):
- Shendure: obs NB
- Cohen: obsingle NB
- Seelig: CM NB
- Takeshi: obs NB

In [ ]:
from pathlib import Path
from IPython.display import display, SVG, Markdown

QC_ROOT = Path(".")  # notebook lives in qc/

# All ortho dirs, grouped by dataset
DATASETS = {
    "Shendure": [
        ("obs NB (canonical)", "canonical_shendure_obs_nb_phantom"),
        ("obs ZINB", "shendure_obs_zinb_phantom"),
        ("CM NB", "shendure_cm_nb_phantom"),
        ("CM ZINB", "shendure_cm_zinb_phantom"),
    ],
    "Cohen": [
        ("obsingle NB (canonical)", "cohen_obsingle_nb_phantom"),
        ("obsingle ZINB", "cohen_obsingle_zinb_phantom"),
        ("obs NB", "cohen_obs_nb_phantom_20260401"),
        ("obs ZINB", "canonical_cohen_obs_zinb_phantom_20260401"),
        ("CM NB", "cohen_cm_nb_phantom_20260401"),
        ("CM ZINB", "cohen_cm_zinb_phantom_20260401"),
    ],
    "Seelig": [
        ("CM NB (canonical)", "canonical_seelig_cm_nb_phantom"),
        ("CM ZINB", "seelig_cm_zinb_phantom"),
    ],
    "Takeshi": [
        ("obs NB (canonical)", "canonical_takeshi_obs_nb_phantom"),
        ("obs ZINB", "takeshi_obs_zinb_phantom"),
        ("CM NB", "takeshi_cm_nb_phantom"),
        ("CM ZINB", "takeshi_cm_zinb_phantom"),
    ],
}

# Plot types to show (in order), excluding worst
PLOTS = [
    "r_values",
    "mu_vs_mean_by_cell_type",
    "theta_distributions",
    "zi_by_cell_type",
    "r_vs_missing",
]

In [ ]:
for dataset, orthos in DATASETS.items():
    display(Markdown(f"---\n# {dataset}"))
    for label, dirname in orthos:
        display(Markdown(f"## {label}\n`{dirname}`"))
        # Show summary.txt snippet
        summary_path = QC_ROOT / dirname / "summary.txt"
        if summary_path.exists():
            txt = summary_path.read_text()
            # Just the convergence + mu range + pearson header lines
            lines = txt.strip().split("\n")
            # Find convergence through end of pearson r section header
            keep = []
            in_section = False
            for line in lines:
                if "Convergence" in line or "Mu range" in line or "Pearson r" in line:
                    in_section = True
                if in_section:
                    keep.append(line)
                    if line.strip().startswith("by_cre: mean r="):
                        keep.append("")  # stop after by_cre summary line
                        in_section = False
            if keep:
                display(Markdown("```\n" + "\n".join(keep) + "\n```"))
        # Show plots
        for plot_name in PLOTS:
            svg_path = QC_ROOT / dirname / "plots" / f"{plot_name}.svg"
            if svg_path.exists():
                display(Markdown(f"### {plot_name.replace('_', ' ').title()}"))
                display(SVG(filename=str(svg_path)))